# pypromptr examples

This notebook shows common ways to use **pypromptr**: a prompt registry with swappable backends (in-memory, markdown files, optional LangFuse).

Run cells top to bottom. Requires `pypromptr` installed (`uv sync` or `pip install -e .` from the repo root).

## 1. In-memory registry

The simplest setup: build a dict of prompts and wrap it in `PromptRegistry`.

In [9]:
from pydantic import BaseModel

from pypromptr import InMemoryPrompts, PromptRegistry, SimplePromptTemplate

class MinimalMetadata(BaseModel):
    pass

registry = PromptRegistry(InMemoryPrompts({
    "welcome": SimplePromptTemplate(
        metadata=MinimalMetadata(),
        content="Hello, {{name}}! Welcome to pypromptr.",
    ),
}))

print("Available keys:", registry.keys())
rendered = registry.get("welcome", name="Alice")
print("Rendered content:", rendered.content)

Available keys: ['welcome']
Rendered content: Hello, Alice! Welcome to pypromptr.


## 2. Prompt types

pypromptr supports four prompt shapes. All expose a `.render(**variables)` method that returns a `RenderedPrompt` (with `content`, `messages`, and `metadata`).

In [10]:
%%time

from pypromptr import (
    MessagesPrompt,
    MessagesPromptTemplate,
    SimplePrompt,
    SimplePromptTemplate,
)

meta = MinimalMetadata()

# Static single-block prompt
static = SimplePrompt(metadata=meta, content="You are a helpful assistant.")
print("SimplePrompt:", static.render().content)

# Single-block with {{placeholders}}
template = SimplePromptTemplate(metadata=meta, content="Summarize this in {{language}}: {{text}}")
print("SimplePromptTemplate:", template.render(language="French", text="Hello world").content)

# Static multi-message prompt (chat format)
messages = MessagesPrompt(
    metadata=meta,
    messages=[
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Explain recursion briefly."},
    ],
)
print("MessagesPrompt roles:", [m["role"] for m in messages.render().messages])

# Multi-message with placeholders
chat_template = MessagesPromptTemplate(
    metadata=meta,
    messages=[
        {"role": "system", "content": "You are a {{role}}."},
        {"role": "user", "content": "Help me with {{task}}."},
    ],
)
rendered_chat = chat_template.render(role="tutor", task="fractions")
for msg in rendered_chat.messages:
    print(f"  {msg['role']}: {msg['content']}")

SimplePrompt: You are a helpful assistant.
SimplePromptTemplate: Summarize this in French: Hello world
MessagesPrompt roles: ['system', 'user']
  system: You are a tutor.
  user: Help me with fractions.
CPU times: user 138 μs, sys: 32 μs, total: 170 μs
Wall time: 153 μs


## 3. File-based prompts

Point `FilePrompts` at a directory of markdown files. Keys come from filenames (without extension). Bodies load lazily on first access.

Supported file formats:
- Plain text body → `SimplePrompt`
- YAML frontmatter + body → metadata on the prompt
- `## system` / `## user` / `## assistant` sections → `MessagesPrompt`

File-loaded prompts are static. Use `SimplePromptTemplate` / `MessagesPromptTemplate` in memory (or a custom source) when you need `{{placeholder}}` substitution.

In [11]:
%%time

import tempfile
from pathlib import Path

from pypromptr import FilePrompts, MessagesPrompt, PromptRegistry, SimplePrompt

# Create sample prompt files in a temp directory
prompts_dir = Path(tempfile.mkdtemp(prefix="pypromptr-examples-"))

(prompts_dir / "greeting.md").write_text(
    "---\ntitle: Greeting\ndescription: A simple welcome prompt\n---\nHello from a markdown file!\n"
)

(prompts_dir / "chat.md").write_text(
    """## system
You are a helpful assistant.

## user
Explain recursion briefly.
"""
)

source = FilePrompts(prompts_dir)
registry = PromptRegistry(source)

print("Keys from filenames:", registry.keys())

greeting = registry.get("greeting")
print("Greeting content:", greeting.content)
print("Greeting metadata title:", greeting.metadata.title)

chat = registry.get_prompt("chat")
assert isinstance(chat, MessagesPrompt)
print("Chat messages:", chat.messages)

Keys from filenames: ['chat', 'greeting']
Greeting content: Hello from a markdown file!
Greeting metadata title: Greeting
Chat messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Explain recursion briefly.'}]
CPU times: user 628 μs, sys: 1.32 ms, total: 1.95 ms
Wall time: 1.46 ms


## 4. Fallback source

Use a primary source (e.g. files) and fall back to in-memory defaults when a key is missing.

In [12]:
%%time

defaults = InMemoryPrompts({
    "fallback-only": SimplePromptTemplate(
        metadata=MinimalMetadata(),
        content="This key only exists in the fallback.",
    ),
})

registry = PromptRegistry(FilePrompts(prompts_dir), defaults)

# From files
print(registry.get("greeting").content)

# From fallback (supports {{placeholders}} via SimplePromptTemplate)
print(registry.get("fallback-only").content)

Hello from a markdown file!
This key only exists in the fallback.
CPU times: user 401 μs, sys: 496 μs, total: 897 μs
Wall time: 582 μs


## 5. Combining registries

Merge two registries with `+`. Resolution is first-wins: keys in the left registry take precedence over the right.

In [13]:
%%time

from pypromptr import SimplePrompt

app_prompts = PromptRegistry(InMemoryPrompts({
    "app-specific": SimplePrompt(metadata=MinimalMetadata(), content="App prompt"),
    "shared": SimplePrompt(metadata=MinimalMetadata(), content="App wins"),
}))

lib_prompts = PromptRegistry(InMemoryPrompts({
    "lib-helper": SimplePrompt(metadata=MinimalMetadata(), content="Library prompt"),
    "shared": SimplePrompt(metadata=MinimalMetadata(), content="Lib version"),
}))

combined = app_prompts + lib_prompts
print("Combined keys:", combined.keys())
print("shared resolves to:", combined.get_prompt("shared").content)
print("lib-helper resolves to:", combined.get_prompt("lib-helper").content)

Combined keys: ['app-specific', 'lib-helper', 'shared']
shared resolves to: App wins
lib-helper resolves to: Library prompt
CPU times: user 187 μs, sys: 74 μs, total: 261 μs
Wall time: 209 μs


## 6. Module-level default registry

For apps that prefer a global registry, use the top-level helpers. Register prompts or whole sources once at startup, then call `get` / `get_prompt` anywhere.

`clear()` resets the default registry (handy in tests).

In [14]:
%%time

import pypromptr
from pypromptr import SimplePrompt, SimplePromptTemplate

pypromptr.clear()  # start fresh

# Register a single prompt override
pypromptr.register_prompt(
    "quick-tip",
    SimplePrompt(metadata=MinimalMetadata(), content="Keep prompts in version control."),
)

# Register a whole source (last registered = highest priority)
pypromptr.register_prompt_source(InMemoryPrompts({
    "onboarding": SimplePromptTemplate(
        metadata=MinimalMetadata(),
        content="Welcome, {{name}}! Start by reading the docs.",
    ),
}))

print(pypromptr.get("onboarding", name="Dev").content)
print(pypromptr.get_prompt("quick-tip").content)

pypromptr.clear()  # clean up for other examples

Welcome, Dev! Start by reading the docs.
Keep prompts in version control.
CPU times: user 133 μs, sys: 71 μs, total: 204 μs
Wall time: 270 μs


## 7. Async access

Use `aget` / `aget_prompt` in async code. Blocking I/O (file reads, remote fetches) runs in a thread pool so your event loop stays responsive.

In [15]:
import asyncio

registry = PromptRegistry(InMemoryPrompts({
    "greeting": SimplePromptTemplate(
        metadata=MinimalMetadata(),
        content="Hello, {{name}}!",
    ),
}))

async def main() -> None:
    prompt = await registry.aget_prompt("greeting")
    rendered = await registry.aget("greeting", name="Async Alice")
    print("Prompt type:", type(prompt).__name__)
    print("Rendered:", rendered.content)

await main()

Prompt type: SimplePromptTemplate
Rendered: Hello, Async Alice!


## 8. Optional: LangFuse backend

Install the extra and configure credentials via environment variables:

```bash
pip install pypromptr[langfuse]
export LANGFUSE_SECRET_KEY=...
export LANGFUSE_PUBLIC_KEY=...
export LANGFUSE_HOST=https://cloud.langfuse.com  # optional
```

Then use `LangFusePrompts` like any other source. Uncomment and adapt the cell below if you have a LangFuse project set up.

In [16]:
# from pypromptr import PromptRegistry
# from pypromptr.langfuse_source import LangFusePrompts
#
# registry = PromptRegistry(LangFusePrompts())
# rendered = registry.get("my-prompt-name", variable="value")
# print(rendered.content)

## 9. CallablePromptSource

Wrap any callable (sync or async) as a prompt source. Useful for computed prompts or mocking in tests.

In [ ]:
import asyncio
from pypromptr import CallablePromptSource, PromptRegistry, SimplePrompt, SimplePromptTemplate
from pydantic import BaseModel

class MinimalMetadata(BaseModel):
    pass

meta = MinimalMetadata()

# Sync callables — zero-argument, return a PromptLike
registry = PromptRegistry(CallablePromptSource({
    "greeting": lambda: SimplePrompt(metadata=meta, content="Hello from a callable!"),
    "template": lambda: SimplePromptTemplate(metadata=meta, content="Hello, {{name}}!"),
}))

print("Keys:", registry.keys())
print(registry.get_prompt("greeting").content)
print(registry.get("template", name="Alice").content)

# Async callable — awaited natively in aget_prompt
async def fetch_system_prompt() -> SimplePrompt:
    return SimplePrompt(metadata=meta, content="You are a helpful assistant.")

registry2 = PromptRegistry(CallablePromptSource({
    "system": fetch_system_prompt,
}))

rendered = await registry2.aget_prompt("system")
print("Async:", rendered.content)

## 10. HttpPrompts

Map prompt keys to HTTP endpoints. Each prompt is fetched on demand.

Install the extra: `pip install pypromptr[http]`

The demo below uses an in-notebook `DemoHttpAdapter` so it runs without installing httpx.

In [ ]:
from pypromptr import PromptRegistry
from pypromptr.sources.http import HttpPrompt, HttpPrompts


class DemoHttpAdapter:
    """In-notebook fake adapter — no httpx required for this demo."""

    def fetch(self, url, method, headers, timeout):
        responses = {
            "https://prompts.example.com/greeting": "Hello, {{name}}!",
            "https://prompts.example.com/chat": "## system\nYou are helpful.\n\n## user\nHello!",
        }
        if url not in responses:
            raise KeyError(url)
        return responses[url]


registry = PromptRegistry(
    HttpPrompts(
        {
            "greeting": HttpPrompt("https://prompts.example.com/greeting"),
            "chat": HttpPrompt("https://prompts.example.com/chat"),
        },
        adapter=DemoHttpAdapter(),
    )
)

print("Keys:", registry.keys())
print("Rendered greeting:", registry.get("greeting", name="Alice").content)

chat = registry.get_prompt("chat")
print("Chat messages:", chat.messages)

## 11. S3Prompts

Fetch prompts from any S3-compatible object store (AWS S3, Cloudflare R2, MinIO, etc.).

Install the extra: `pip install pypromptr[s3]`

The demo below uses an in-notebook `FakeBucketAdapter` so it runs without AWS credentials.

In [ ]:
from pypromptr import PromptRegistry
from pypromptr.sources.s3 import S3Prompts


class FakeBucketAdapter:
    """In-notebook fake adapter — no boto3 or AWS credentials required for this demo."""

    def __init__(self, objects):
        self._objects = objects

    def list_keys(self):
        return list(self._objects.keys())

    def get_object(self, key):
        if key not in self._objects:
            raise KeyError(key)
        return self._objects[key]


registry = PromptRegistry(
    S3Prompts(
        adapter=FakeBucketAdapter({
            "welcome": "---\ntitle: Welcome\n---\nHello from S3!",
            "chat":    "## system\nYou are a helpful assistant.\n\n## user\nHow can I help you?",
        })
    )
)

print("Keys:", registry.keys())

welcome = registry.get_prompt("welcome")
print("Welcome content:", welcome.content)
print("Welcome title:", welcome.metadata.model_extra.get("title"))

chat = registry.get_prompt("chat")
print("Chat messages:", chat.messages)

# Real usage (requires pip install pypromptr[s3] and AWS credentials):
# registry = PromptRegistry(S3Prompts("my-bucket", prefix="prompts/prod/"))

## 12. GitRepoPrompts

Load prompts from a Git repository at any branch, tag, or commit SHA.

Install the extra: `pip install pypromptr[git]`

The demo below uses an in-notebook `FakeGitAdapter` so it runs without cloning a repo.

In [ ]:
from pypromptr import PromptRegistry
from pypromptr.sources.git import GitRepoPrompts


class FakeGitAdapter:
    """In-notebook fake adapter — no gitpython or network required for this demo."""

    def __init__(self, files):
        self._files = files

    def list_keys(self):
        return list(self._files.keys())

    def read_file(self, key):
        if key not in self._files:
            raise KeyError(key)
        return self._files[key]


registry = PromptRegistry(
    GitRepoPrompts(
        adapter=FakeGitAdapter({
            "system-prompt": "---\ntitle: System\n---\nYou are a helpful assistant.",
            "user-template": "Hello, {{name}}! How can I assist you today?",
        })
    )
)

print("Keys:", registry.keys())
system = registry.get_prompt("system-prompt")
print("System content:", system.content)
print("System title:", system.metadata.model_extra.get("title"))

# Real usage (requires pip install pypromptr[git]):
# registry = PromptRegistry(
#     GitRepoPrompts(
#         url="https://github.com/org/prompts.git",
#         ref="v2.0.0",
#         path="prompts/",
#     )
# )

## 13. SqlPrompts

Query prompts from any SQLAlchemy-supported database (PostgreSQL, MySQL, SQLite, etc.).

Install the extra: `pip install pypromptr[sql]`

The demo below uses an in-memory SQLite database — no external DB needed.

In [ ]:
import json
from sqlalchemy import create_engine, text

from pypromptr import PromptRegistry
from pypromptr.sources.sql import SqlPrompts

# Set up an in-memory SQLite database
engine = create_engine("sqlite:///:memory:", future=True)
with engine.begin() as conn:
    conn.execute(text(
        "CREATE TABLE prompts (key TEXT PRIMARY KEY, content TEXT NOT NULL, metadata TEXT)"
    ))
    conn.execute(text(
        "INSERT INTO prompts VALUES ('greeting', 'Hello, {{name}}!', :meta)"
    ), {"meta": json.dumps({"title": "Greeting"})})
    conn.execute(text(
        "INSERT INTO prompts VALUES ('system', '## system\nYou are helpful.\n\n## user\nHi!', NULL)"
    ))
    conn.execute(text(
        "INSERT INTO prompts VALUES ('static', 'This is a static prompt.', NULL)"
    ))

registry = PromptRegistry(SqlPrompts(engine=engine))

print("Keys:", registry.keys())

rendered = registry.get("greeting", name="Alice")
print("Greeting:", rendered.content)

chat = registry.get_prompt("system")
print("Chat messages:", chat.messages)

static = registry.get_prompt("static")
print("Static content:", static.content)

try:
    registry.get_prompt("nonexistent")
except KeyError as e:
    print("KeyError raised:", e)